# How Timeseries Aggregation works

Time-series aggregation (TSA) reduces the **temporal complexity** of a time series so a
downstream optimisation model can afford to solve over it. There are two broad ways to do
this: lower the **resolution** (fewer or coarser time steps over the same calendar), or
represent the series with a few **typical periods** (representative period-shapes carrying
occurrence weights). This overview starts from that general picture, narrows to tsam's
specific building blocks, and maps the rest of the series — every step of which is traced by
hand on a tiny six-day dataset (introduced in [Preprocessing](01_preprocessing.ipynb)).

## 1  The TSA taxonomy: two axes of common methods

[Hoffmann et al. (2020)](https://www.mdpi.com/1996-1073/13/3/641) classifies common TSA methods along two independent axes that are shown in the table below:

* **How** periods or timesteps are grouped — by **time position** (consecutive blocks,
  calendar position) or by **feature similarity** (grouping by value).
* **What the result is** — **resolution variation** (fewer/coarser timesteps, same calendar)
  or **typical periods** (a few representative period-shapes with occurrence weights).

A ✅ marks what tsam implements:

|                   | **Resolution variation** | **Typical periods**          |
|-------------------|--------------------------|------------------------------|
| **Time-based**    | Downsampling *(use pandas)* | Averaging ✅ *(consecutive-period blocks; full calendar time-slices not built in)* |
| **Feature-based** | Segmentation ✅          | **Clustering** ✅ *(core)*   |

tsam focuses on the **feature-based column** (clustering and segmentation) plus time-based block averaging.

Two nuances worth fixing early, because the names mislead. `contiguous` is *not* time-based
averaging: it is Ward clustering with a temporal-adjacency constraint — **feature-based**, and
the period-level analogue of segmentation (only adjacent periods may merge). `averaging` is
the one genuinely **time-based** grouping here: consecutive blocks by position, ignoring
values.

*Methods that fall outside this 2x2 grid* are currently out of tsam's scope. However tsam is open to collaborations an contributions. Further methods include e.g. : shape- and time-shift-tolerant clustering (dynamic time warping, k-shape), dimensionality-reduction pre-processing (PCA, autoencoders), multiple time grids per season, random period sampling, and full calendar time-slices(which need an external assignment vector). Plain downsampling is a one-liner in pandas (`df.resample(rule).mean()`) and is
intentionally not duplicated. tsam also never performs *cross-sectional grouping of time
series*: it preserves the input's dimensionality, so an `N`-attribute series stays
`N`-attribute throughout.

## 2  How tsam decomposes the clustering step

[Hoffmann et al. (2020)](https://www.mdpi.com/1996-1073/13/3/641) decompose feature-based time-series clustering into **five
fundamental aspects** — and tsam exposes each one as an independent choice:

> This means that time series clustering includes five fundamental aspects:
>
> - A normalization (and sometimes a dimensionality reduction).
> - A distance metric.
> - A clustering algorithm.
> - A method to choose representatives.
> - A rescaling step in the case of non-centroid based clustering algorithms.
>

This structure shapes the architecture of tsam and the guide in the following. **Clustering algorithm** (which periods are
grouped together) and the **method to choose representatives** (how each group becomes one
profile) are *separate, recombinable* steps. Classical algorithms tie the two together:
k-means uses the mean, k-medoids the medoid . Tsam lets you configure these freely to let you tailor your time series aggregation to your needs.

Those recombinable axes, and the optional steps that hang off them, are the whole pipeline in
one picture:

![The aggregation pipeline: grouping x representation, plus the optional post-steps](../../assets/architecture/aggregation_methods.svg)

## 3  What tsam implements: clustering x representation

The first below shows the clustering methods implemented in tsam and their respective standard representation. The second table shows all implemented representations.

**Axis 1 — Grouping (clustering) methods** *(which periods go together)*

All clustering methods live together under **section 2, `02_clustering/`** — the three
feature-based paradigms (labelled 2.1–2.3 below) plus time-based block averaging (2.4).

| Clustering method | tsam name | Default representation | Notebook |
|---|---|---|---|
| k-means | `kmeans` | mean | [2.1](02_clustering/01_partitional_clustering.ipynb) |
| k-medoids | `kmedoids` | medoid | [2.1](02_clustering/01_partitional_clustering.ipynb) |
| k-maxoids&nbsp; | `kmaxoids` | maxoid | [2.3](02_clustering/03_extremal_prototype_selection.ipynb) |
| Hierarchical Ward | `hierarchical` | medoid | [2.2](02_clustering/02_agglomerative_clustering.ipynb) |
| Contiguous Ward | `contiguous` | medoid | [2.2](02_clustering/02_agglomerative_clustering.ipynb) |
| Block averaging | `averaging` | mean | [2.4](02_clustering/04_averaging.ipynb) |

**k-maxoids optimises a different objective** from the other partitional methods — the spread
*between* representatives, not the within-cluster distance $J$.
See **three clustering paradigms** below.

**Axis 2 — Representation methods** *(how each group becomes one profile)*

| Representation | tsam name | What it produces | Notebook |
|---|---|---|---|
| Centroid | `mean` | per-timestep average of the group  | [03](03_representation.ipynb) |
| Medoid | `medoid` | the most central real period  | [03](03_representation.ipynb) |
| Maxoid | `maxoid` | the most extreme real period  | [03](03_representation.ipynb) |
| Distribution | `distribution` | values re-sorted to keep the duration curve  | [03](03_representation.ipynb) |
| Distribution + min/max | `distribution_minmax` | duration curve with per-column min/max preserved | [03](03_representation.ipynb) |
| Min/max + mean | `minmax_mean` | mean with per-column extremes preserved |  [03](03_representation.ipynb) |

> **Two senses of "maxoid".** The k-maxoids **clustering** method (Axis 1) *selects*
> spread-maximal periods while grouping; the maxoid **representation** (Axis 2) *picks* the most
> extreme member of an already-formed cluster. Same word, different pipeline steps — you can use
> the maxoid representation with any clustering (e.g. Ward + maxoid).

In addition to the clustering and representation steps, the following optional methods can be
used to adapt the aggregation to your needs. They are ordered here as the pipeline applies them
— extremes and rescaling in the clustering phase, segmentation last:

* **Extreme periods** (`ExtremeConfig`) — a *clustering extension* applied right after
  clustering: inject peak/trough periods into the cluster set (`append` / `replace` /
  `new_cluster`) so they survive the aggregation. See [04](04_extreme_periods.ipynb).
* **Rescaling** (`preserve_column_means`) — a *representation post-step* (the review's fifth
  aspect): restore column means after a non-centroid representation. See [05](05_rescaling.ipynb).
* **Segmentation** (`SegmentConfig`) — the *resolution-variation axis*, applied last to the
  formed typical periods: merge adjacent timesteps into fewer segments. See [06](06_segmentation.ipynb).

*Natural extension points* — methods that fit this same grouping x representation pipeline
and could be added without architectural changes: k-medians and k-centers, other
agglomerative linkages (single / complete / average), additional distance methods (L1), and
further representation methods.

## 4  Three clustering paradigms

The **feature-based** grouping methods in Axis 1 fall into three families that optimise
**different objectives** (block `averaging`, being time-based, sits outside them). It is worth
keeping them apart, because only the first two chase the within-cluster distance
$J$ that most of this guide builds on:

| Paradigm | tsam methods | What it optimises | Representative sits… | The partition is… |
|---|---|---|---|---|
| **Partitional** | `kmeans`, `kmedoids` | within-cluster distance $J$ (over the *assignment*) | in the **middle** of its group | *what is optimised* |
| **Agglomerative** | `hierarchical`, `contiguous` | the increase in within-cluster variance at each greedy merge | in the **middle** of its group | *what is optimised* (bottom-up) |
| **Extremal-prototype** | `kmaxoids` | the spread $E(\mathcal{Z})$ *between* representatives (no assignment term) | on the **convex hull / extremes** | a nearest-assignment **by-product** |

Partitional and agglomerative clustering both pull representatives toward cluster centres; they
differ mainly in *how* the groups are built — iterative refinement versus bottom-up merging.
Extremal-prototype selection (k-maxoids) is the odd one out: it never looks at cluster
membership while choosing representatives, and forms a partition only afterwards by assigning
each period to its nearest prototype. Each paradigm has its own notebook —
[Partitional clustering](02_clustering/01_partitional_clustering.ipynb),
[Agglomerative clustering](02_clustering/02_agglomerative_clustering.ipynb) and
[Extremal-prototype selection](02_clustering/03_extremal_prototype_selection.ipynb).

## How this series is organised

Each notebook goes deep on one part of the pipeline above, all on the same tiny six-day
dataset, and in the order the pipeline applies them:

1. [Preprocessing](01_preprocessing.ipynb) — normalization and unstacking to the period matrix (and a first look at the dataset).
2. **Clustering** (`02_clustering/`) — four ways to group the periods:
    1. [Partitional clustering](02_clustering/01_partitional_clustering.ipynb) — k-means and k-medoids.
    2. [Agglomerative clustering](02_clustering/02_agglomerative_clustering.ipynb) — hierarchical and contiguous Ward.
    3. [Extremal-prototype selection](02_clustering/03_extremal_prototype_selection.ipynb) — k-maxoids, the spread-maximising method.
    4. [Averaging](02_clustering/04_averaging.ipynb) — positional block grouping (the one time-based method).
3. [Representation](03_representation.ipynb) — mean / medoid / maxoid / distribution strategies for turning each cluster into one profile.
4. [Extreme periods](04_extreme_periods.ipynb) — append / replace / new_cluster strategies that inject peaks into the cluster set.
5. [Rescaling](05_rescaling.ipynb) — restoring totals and denormalising to physical units.
6. [Segmentation](06_segmentation.ipynb) — fewer timesteps within each typical period.

### Further reading

* [Notation and equations](../../reference/notation.md) — every symbol and formula on one page
* [Pipeline Guide](../background/architecture/pipeline_guide.md) — the four pipeline phases
* [Clustering methods](../../how-to/clustering_methods.ipynb) — accuracy and speed benchmarks